# LSTM Signal Predictor Training Demo

This notebook demonstrates training an LSTM model for binary trading signal prediction (hold=0, trade=1).

## Components:
- **Data**: `doge.csv` - Binary indicator signals
- **Model**: `LSTMSignalPredictor` - LSTM-based classifier
- **Trainer**: Handles training loop, validation, early stopping
- **Loss**: Weighted CrossEntropy with optional focal loss

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import torch

# Import LSTM modules
from crypto_analysis.lstm.model import ModelConfig, LSTMSignalPredictor
from crypto_analysis.lstm.trainer import Trainer, TrainingConfig
from crypto_analysis.lstm.loss import BinarySignalLoss, FocalBinaryLoss
from crypto_analysis.lstm.data_preprocessor import DataPreprocessor
from crypto_analysis.lstm.dataset import SignalDataset, create_sequences

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Load and Explore Data

In [ ]:
# Load the dataframe
df = pd.read_csv("doge.csv")

print(f"DataFrame shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
print(df.columns.tolist()[:20], "...")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Check target distribution
print("Target (tradeable) distribution:")
print(df['tradeable'].value_counts())
print(f"\nClass ratio: {df['tradeable'].value_counts(normalize=True).to_dict()}")

## 2. Data Preprocessing

In [ ]:
# Define excluded columns (metadata, not features)
EXCLUDED_COLUMNS = ['date', 'signal', 'signal_pct_change', 'period_id', 'tradeable']

# Get feature columns
feature_columns = [col for col in df.columns if col not in EXCLUDED_COLUMNS]
print(f"Number of feature columns: {len(feature_columns)}")
print(f"Sample features: {feature_columns[:10]}")

In [ ]:
# Prepare dataframe for preprocessing (keep tradeable + features)
df_selected = df[['tradeable'] + feature_columns].copy()

# Align dataframe for period-consistent sequences
df_aligned = DataPreprocessor.align_dataframe(df_selected, period_size=4, verbose=True)

print(f"\nAligned DataFrame shape: {df_aligned.shape}")

In [ ]:
# Initialize preprocessor and transform data
preprocessor = DataPreprocessor(target_shift=4)
features, targets = preprocessor.fit_transform(df_aligned)

print(f"Features shape: {features.shape}")
print(f"Targets shape: {targets.shape}")
print(f"Unique targets: {np.unique(targets)}")

## 3. Create Sequences and Dataset

In [ ]:
# Create sequences
INPUT_SEQ_LENGTH = 24  # Number of timesteps to look back
OUTPUT_SEQ_LENGTH = 1   # Binary prediction (single output)
STRIDE = 4              # Period-aligned stride

feat_seqs, tgt_seqs = create_sequences(
    features, targets,
    input_seq_length=INPUT_SEQ_LENGTH,
    output_seq_length=OUTPUT_SEQ_LENGTH,
    stride=STRIDE
)

print(f"Feature sequences shape: {feat_seqs.shape}")
print(f"Target sequences shape: {tgt_seqs.shape}")

In [ ]:
# Verify no mixed labels in target sequences
mixed_count = 0
for i, tgt in enumerate(tgt_seqs):
    if len(np.unique(tgt)) > 1:
        mixed_count += 1

print(f"Sequences with mixed labels: {mixed_count}/{len(tgt_seqs)}")

# Check label distribution
hold_count = (tgt_seqs.flatten() == 0).sum()
trade_count = (tgt_seqs.flatten() == 1).sum()
print(f"Label distribution: hold={hold_count}, trade={trade_count}")

In [ ]:
# Create dataset
dataset = SignalDataset(feat_seqs, tgt_seqs)
print(f"Dataset size: {len(dataset)}")

# Inspect a sample
sample = dataset[2]
print(f"\nSample features shape: {sample['features'].shape}")
print(f"Sample target shape: {sample['targets'].shape}")
print(f"Sample target value: {sample['targets']}")

## 4. Model Configuration

In [ ]:
# Model configuration
model_config = ModelConfig(
    input_size=preprocessor.get_num_features(),
    hidden_size=128,
    num_layers=2,
    classifier_hidden_size=64,
    dropout=0.2,
    bidirectional=False,
    input_seq_length=INPUT_SEQ_LENGTH,
)

print(f"Model config:")
print(f"  input_size: {model_config.input_size}")
print(f"  hidden_size: {model_config.hidden_size}")
print(f"  num_layers: {model_config.num_layers}")
print(f"  classifier_hidden_size: {model_config.classifier_hidden_size}")
print(f"  dropout: {model_config.dropout}")

In [ ]:
# Create model
model = LSTMSignalPredictor(model_config)
print(model)
print(f"\nTotal parameters: {model.get_num_parameters():,}")

In [ ]:
# Test forward pass
test_input = torch.randn(4, INPUT_SEQ_LENGTH, model_config.input_size)
test_output = model(test_input)
print(f"Test input shape: {test_input.shape}")
print(f"Test output shape: {test_output.shape}")
print(f"Test output (logits): {test_output[0]}")

## 5. Training Configuration

In [ ]:
# Training configuration
training_config = TrainingConfig(
    # Model architecture (used if creating model from config)
    hidden_size=128,
    num_layers=2,
    dropout=0.2,
    
    # Training parameters
    epochs=200,
    batch_size=16,
    learning_rate=1e-1,
    weight_decay=1e-3,
    optimizer='adam',
    grad_clip_norm=1.0,
    
    # Learning rate scheduler
    scheduler='plateau',
    scheduler_patience=10,
    scheduler_factor=0.75,
    
    # Class imbalance handling
    auto_class_weights=True,
    class_weight_power=0.5,
    focal_loss=False,
    focal_gamma=2.0,
    label_smoothing=0.05,
    
    # Data split
    val_split=0.2,
    test_split=0.2,
    
    # Early stopping
    early_stopping=False,
    patience=15,
    min_delta=1e-4,
    
    # Device
    device='auto',
    
    # Logging
    log_interval=50,
    verbose=True,
)

print("Training configuration created.")

## 6. Initialize Trainer

In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    config=training_config,
    preprocessor=preprocessor  # Save preprocessor with checkpoints
)

print(trainer)

## 7. Train the Model

In [ ]:
# Train the model
history = trainer.train(dataset)

print(f"\nTraining complete!")
print(f"Best epoch: {history.best_epoch + 1}")
print(f"Best validation loss: {history.best_val_loss:.4f}")

## 8. Training History Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss curves
axes[0].plot(history.train_losses, label='Train Loss')
axes[0].plot(history.val_losses, label='Val Loss')
axes[0].axvline(x=history.best_epoch, color='r', linestyle='--', label='Best Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy curves
axes[1].plot(history.train_accuracies, label='Train Acc')
axes[1].plot(history.val_accuracies, label='Val Acc')
axes[1].axvline(x=history.best_epoch, color='r', linestyle='--', label='Best Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

# Learning rate
axes[2].plot(history.learning_rates)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].set_yscale('log')
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 9. Model Evaluation

In [ ]:
# Evaluate on all datasets (train, val, test)
results = trainer.evaluate_all(verbose=True)

In [ ]:
# Visualize confusion matrices
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, metrics) in enumerate(results.items()):
    if metrics is None:
        continue
    
    cm = np.array(metrics['confusion_matrix'])
    sns.heatmap(
        cm, 
        annot=True, 
        fmt='d', 
        cmap='Blues',
        xticklabels=['hold', 'trade'],
        yticklabels=['hold', 'trade'],
        ax=axes[idx]
    )
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_title(f'{name.upper()} Confusion Matrix\nF1: {metrics["f1"]:.3f}')

plt.tight_layout()
plt.show()

## 10. Make Predictions

In [ ]:
# Get predictions on test set
if trainer.test_dataset is not None:
    from torch.utils.data import DataLoader
    
    test_loader = DataLoader(
        trainer.test_dataset,
        batch_size=32,
        shuffle=False,
        collate_fn=SignalDataset.collate_fn
    )
    
    all_preds = []
    all_probs = []
    all_targets = []
    
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            features = batch['features'].to(trainer.device)
            targets = batch['targets']
            
            logits = model(features)
            probs = torch.softmax(logits, dim=-1)
            preds = torch.argmax(logits, dim=-1)
            
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())  # Trade probability
            all_targets.extend(targets.numpy())
    
    print(f"Test set predictions: {len(all_preds)}")
    print(f"\nSample predictions (first 10):")
    for i in range(min(10, len(all_preds))):
        label = 'trade' if all_targets[i] == 1 else 'hold'
        pred_label = 'trade' if all_preds[i] == 1 else 'hold'
        correct = '✓' if all_preds[i] == all_targets[i] else '✗'
        print(f"  {i+1}. Actual: {label:<5} | Pred: {pred_label:<5} | Trade prob: {all_probs[i]:.3f} | {correct}")

## 11. Loss Function Examples

In [ ]:
# Demonstrate different loss functions

# 1. Basic Binary Signal Loss (weighted CrossEntropy)
class_weights = torch.tensor([1.0, 3.0])  # Weight trade class higher
basic_loss = BinarySignalLoss(class_weights=class_weights, label_smoothing=0.05)
print(f"BinarySignalLoss: {basic_loss}")
print(f"  Config: {basic_loss.get_weight_summary()}")

# 2. Focal Loss (for hard example mining)
focal_loss = FocalBinaryLoss(gamma=2.0, class_weights=class_weights, label_smoothing=0.05)
print(f"\nFocalBinaryLoss: {focal_loss}")
print(f"  Config: {focal_loss.get_weight_summary()}")

In [ ]:
# Compare loss values on sample data
sample_logits = torch.tensor([[2.0, -1.0], [-1.0, 2.0], [0.5, 0.5], [1.0, 0.0]])
sample_targets = torch.tensor([0, 1, 0, 1])  # hold, trade, hold, trade

print("Sample predictions:")
probs = torch.softmax(sample_logits, dim=-1)
for i, (logit, target) in enumerate(zip(sample_logits, sample_targets)):
    label = 'trade' if target == 1 else 'hold'
    print(f"  {i+1}. Target: {label:<5} | Probs: hold={probs[i,0]:.3f}, trade={probs[i,1]:.3f}")

print(f"\nLoss comparison:")
print(f"  BinarySignalLoss: {basic_loss(sample_logits, sample_targets):.4f}")
print(f"  FocalBinaryLoss:  {focal_loss(sample_logits, sample_targets):.4f}")

## Summary

This notebook demonstrated:

1. **Data Loading**: Loading binary indicator signals from CSV
2. **Preprocessing**: Aligning data for period-consistent sequences
3. **Sequence Creation**: Creating input/output sequences with stride
4. **Model Setup**: Configuring `LSTMSignalPredictor` with custom classifier size
5. **Training**: Using `Trainer` with class weights, early stopping, and LR scheduling
6. **Evaluation**: Per-class metrics and confusion matrices
7. **Loss Functions**: `BinarySignalLoss` and `FocalBinaryLoss` for class imbalance